In [1]:
from pyexpat import model
from sklearn.preprocessing import LabelBinarizer
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np
import glob
import cv2
import os
import locale
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import AveragePooling2D
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Dense, Add, ZeroPadding2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import concatenate
from tensorflow.keras.metrics import Precision, Recall, BinaryAccuracy
from sklearn import preprocessing
import locale
from matplotlib import pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from deepforest import CascadeForestClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

from tensorflow.keras.layers import DepthwiseConv2D
from tensorflow.keras.layers import ReLU, AvgPool2D
# from tensorflow.keras.layers import Input, DepthwiseConv2D
# from tensorflow.keras.layers import Conv2D, BatchNormalization
# from tensorflow.keras.layers import ReLU, AvgPool2D, Flatten, Dense
# from tensorflow.keras import Model

def create_cnn_alexnet():
    model = Sequential()
    model.add(Conv2D(filters = 96, input_shape = (224, 224, 3),  
                kernel_size = (11, 11), strides = (4, 4),  
                padding = 'valid')) 
    model.add(Activation('relu')) 
    # Max-Pooling  
    model.add(MaxPooling2D(pool_size = (2, 2), 
                strides = (2, 2), padding = 'valid')) 
    # Batch Normalisation 
    model.add(BatchNormalization()) 
    
    # 2nd Convolutional Layer 
    model.add(Conv2D(filters = 256, kernel_size = (11, 11),  
                strides = (1, 1), padding = 'valid')) 
    model.add(Activation('relu')) 
    # Max-Pooling 
    model.add(MaxPooling2D(pool_size = (2, 2), strides = (2, 2),  
                padding = 'valid')) 
    # Batch Normalisation 
    model.add(BatchNormalization()) 
    
    # 3rd Convolutional Layer 
    model.add(Conv2D(filters = 384, kernel_size = (3, 3),  
                strides = (1, 1), padding = 'valid')) 
    model.add(Activation('relu')) 
    # Batch Normalisation 
    model.add(BatchNormalization()) 
    
    # 4th Convolutional Layer 
    model.add(Conv2D(filters = 384, kernel_size = (3, 3),  
                strides = (1, 1), padding = 'valid')) 
    model.add(Activation('relu')) 
    # Batch Normalisation 
    model.add(BatchNormalization()) 
    
    # 5th Convolutional Layer 
    model.add(Conv2D(filters = 256, kernel_size = (3, 3),  
                strides = (1, 1), padding = 'valid')) 
    model.add(Activation('relu')) 
    # Max-Pooling 
    model.add(MaxPooling2D(pool_size = (2, 2), strides = (2, 2),  
                padding = 'valid')) 
    # Batch Normalisation 
    model.add(BatchNormalization()) 
    
    # Flattening 
    model.add(Flatten()) 
    
    # 1st Dense Layer 
    model.add(Dense(4096, input_shape = (224*224*3, ))) 
    model.add(Activation('relu')) 
    # Add Dropout to prevent overfitting 
    model.add(Dropout(0.4)) 
    # Batch Normalisation 
    model.add(BatchNormalization()) 
    
    # 2nd Dense Layer 
    model.add(Dense(4096)) 
    model.add(Activation('relu')) 
    # Add Dropout 
    model.add(Dropout(0.4)) 
    # Batch Normalisation 
    model.add(BatchNormalization()) 
    model.add(Dense(2048, activation = 'relu'))
    model.add(Dropout(0.4))
    model.add(Dense(1, activation = 'relu'))
    return model

def identity_block(X, f, filters, stage, block):
   
    conv_name_base = 'res' + str(stage) + block + '_branch'
    bn_name_base = 'bn' + str(stage) + block + '_branch'
    F1, F2, F3 = filters

    X_shortcut = X
   
    X = Conv2D(filters=F1, kernel_size=(1, 1), strides=(1, 1), padding='valid', name=conv_name_base + '2a')(X)
    X = BatchNormalization(axis=3, name=bn_name_base + '2a')(X)
    X = Activation('relu')(X)

    X = Conv2D(filters=F2, kernel_size=(f, f), strides=(1, 1), padding='same', name=conv_name_base + '2b')(X)
    X = BatchNormalization(axis=3, name=bn_name_base + '2b')(X)
    X = Activation('relu')(X)

    X = Conv2D(filters=F3, kernel_size=(1, 1), strides=(1, 1), padding='valid', name=conv_name_base + '2c')(X)
    X = BatchNormalization(axis=3, name=bn_name_base + '2c')(X)
    X = Add()([X, X_shortcut])# SKIP Connection
    X = Activation('relu')(X)

    return X

def convolutional_block(X, f, filters, stage, block, s=2):
   
    conv_name_base = 'res' + str(stage) + block + '_branch'
    bn_name_base = 'bn' + str(stage) + block + '_branch'

    F1, F2, F3 = filters

    X_shortcut = X

    X = Conv2D(filters=F1, kernel_size=(1, 1), strides=(s, s), padding='valid', name=conv_name_base + '2a')(X)
    X = BatchNormalization(axis=3, name=bn_name_base + '2a')(X)
    X = Activation('relu')(X)

    X = Conv2D(filters=F2, kernel_size=(f, f), strides=(1, 1), padding='same', name=conv_name_base + '2b')(X)
    X = BatchNormalization(axis=3, name=bn_name_base + '2b')(X)
    X = Activation('relu')(X)
    X = Conv2D(filters=F3, kernel_size=(1, 1), strides=(1, 1), padding='valid', name=conv_name_base + '2c')(X)
    X = BatchNormalization(axis=3, name=bn_name_base + '2c')(X)
    X_shortcut = Conv2D(filters=F3, kernel_size=(1, 1), strides=(s, s), padding='valid', name=conv_name_base + '1',)(X_shortcut)
    X_shortcut = BatchNormalization(axis=3, name=bn_name_base + '1')         (X_shortcut)

    X = Add()([X, X_shortcut])
    X = Activation('relu')(X)

    return X

def ResNet50(input_shape=(224, 224, 3)):

    X_input = Input(input_shape)

    X = ZeroPadding2D((3, 3))(X_input)

    X = Conv2D(64, (7, 7), strides=(2, 2), name='conv1', )(X)
    X = BatchNormalization(axis=3, name='bn_conv1')(X)
    X = Activation('relu')(X)
    X = MaxPooling2D((3, 3), strides=(2, 2))(X)

    X = convolutional_block(X, f=3, filters=[64, 64, 256], stage=2, block='a', s=1)
    X = identity_block(X, 3, [64, 64, 256], stage=2, block='b')
    X = identity_block(X, 3, [64, 64, 256], stage=2, block='c')


    X = convolutional_block(X, f=3, filters=[128, 128, 512], stage=3, block='a', s=2)
    X = identity_block(X, 3, [128, 128, 512], stage=3, block='b')
    X = identity_block(X, 3, [128, 128, 512], stage=3, block='c')
    X = identity_block(X, 3, [128, 128, 512], stage=3, block='d')
    X = convolutional_block(X, f=3, filters=[256, 256, 1024], stage=4, block='a', s=2)
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block='b')
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block='c')
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block='d')
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block='e')
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block='f')

    X = X = convolutional_block(X, f=3, filters=[512, 512, 2048], stage=5, block='a', s=2)
    X = identity_block(X, 3, [512, 512, 2048], stage=5, block='b')
    X = identity_block(X, 3, [512, 512, 2048], stage=5, block='c')

    X = AveragePooling2D(pool_size=(2, 2), padding='same')(X)
    headModel = Flatten()(X)
    headModel=Dense(256, activation='relu', name='fc1')(headModel)
    headModel=Dense(128, activation='relu', name='fc2')(headModel)
    headModel = Dense( 1, name='fc3')(headModel)
    model = Model(inputs=X_input, outputs=headModel, name='ResNet50')
    return model
# MobileNet block
def mobilnet_block (x, filters, strides):
    
    x = DepthwiseConv2D(kernel_size = 3, strides = strides, padding = 'same')(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    
    x = Conv2D(filters = filters, kernel_size = 1, strides = 1)(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    
    return x

def create_cnn_mobilenet():
    input = Input(shape = (224,224,3))
    x = Conv2D(filters = 32, kernel_size = 3, strides = 2, padding = 'same')(input)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    # main part of the model
    x = mobilnet_block(x, filters = 64, strides = 1)
    x = mobilnet_block(x, filters = 128, strides = 2)
    x = mobilnet_block(x, filters = 128, strides = 1)
    x = mobilnet_block(x, filters = 256, strides = 2)
    x = mobilnet_block(x, filters = 256, strides = 1)
    x = mobilnet_block(x, filters = 512, strides = 2)
    
    for _ in range (5):
        x = mobilnet_block(x, filters = 512, strides = 1)
    x = mobilnet_block(x, filters = 1024, strides = 2)
    x = mobilnet_block(x, filters = 1024, strides = 1)
    x = AvgPool2D (pool_size = 7, strides = 1, data_format='channels_first')(x)
    x = Flatten()(x)
    x = Dense(units = 1024, activation = 'relu')(x)
    output = Dropout(0.4)(x)
    output = Dense (1)(output)
    model = Model(inputs=input, outputs=output)
    # model.summary()

    return model


gasdata = pd.read_csv("/Users/wzhang/Downloads/Gas_Sensors_Measurements (2).csv")
data_dir = '/Users/wzhang/Downloads/NoGas'
imagedata = sorted(os.listdir(data_dir))
print(len(imagedata))
X_data = []
for image in imagedata:
        print(image)
        img = mpimg.imread('/Users/wzhang/Downloads/NoGas/'+image)
        img.resize(224,224,3)
        img = img/255.0
        X_data.append(img)
images = np.array(X_data)
print(images[0])
print(images.shape)

split = train_test_split(gasdata, images, test_size=0.2)
(trainAttrX, testAttrX, trainImagesX, testImagesX) = split
trainy = trainAttrX["Gas"]
testy = testAttrX["Gas"]
trainAttrX = trainAttrX.drop(columns=['Gas'])
testAttrX = testAttrX.drop(columns = ['Gas'])
trainAttrX= (trainAttrX - np.min(trainAttrX)) / (np.max(trainAttrX) - np.min(trainAttrX))
testAttrX = (testAttrX - np.min(testAttrX)) / (np.max(testAttrX) - np.min(testAttrX))
print(trainAttrX.shape)
print(testAttrX.shape)
print(trainy.shape)
print(testy.shape)

alexnet = create_cnn_alexnet()
alexnet.compile(loss = None)
alexnetPredict = alexnet.predict(trainImagesX)
print(alexnetPredict.shape)

resnet = ResNet50()
resnetPredict = resnet.predict(trainImagesX)
print(resnetPredict.shape)

mobilenet = create_cnn_mobilenet()
mobilenetPredict = mobilenet.predict(trainImagesX)
print(mobilenetPredict.shape)

alexnetPredict = np.array(alexnetPredict)
resnetPredict = np.array(resnetPredict)
mobilenetPredict = np.array(mobilenetPredict)
trainAttrX = np.array(trainAttrX)
trainAttrX.resize(5120,2)
trainData = concatenate([alexnetPredict,resnetPredict,mobilenetPredict,trainAttrX])
trainData = np.array(trainData)
trainData.reshape(-1,1)

print(trainData.shape)




2024-07-02 15:59:48.617361: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


6400
A (1).png
A (10).png
A (100).png
A (1000).png
A (1001).png
A (1002).png
A (1003).png
A (1004).png
A (1005).png
A (1006).png
A (1007).png
A (1008).png
A (1009).png
A (101).png
A (1010).png
A (1011).png
A (1012).png
A (1013).png
A (1014).png
A (1015).png
A (1016).png
A (1017).png
A (1018).png
A (1019).png
A (102).png
A (1020).png
A (1021).png
A (1022).png
A (1023).png
A (1024).png
A (1025).png
A (1026).png
A (1027).png
A (1028).png
A (1029).png
A (103).png
A (1030).png
A (1031).png
A (1032).png
A (1033).png
A (1034).png
A (1035).png
A (1036).png
A (1037).png
A (1038).png
A (1039).png
A (104).png
A (1040).png
A (1041).png
A (1042).png
A (1043).png
A (1044).png
A (1045).png
A (1046).png
A (1047).png
A (1048).png
A (1049).png
A (105).png
A (1050).png
A (1051).png
A (1052).png
A (1053).png
A (1054).png
A (1055).png
A (1056).png
A (1057).png
A (1058).png
A (1059).png
A (106).png
A (1060).png
A (1061).png
A (1062).png
A (1063).png
A (1064).png
A (1065).png
A (1066).png
A (1067).png
A (106

In [ ]:
import sktime

In [ ]:
model = CascadeForestClassifier()
from sktime.classification.sklearn import RotationForest
from sklearn.ensemble import RandomForestClassifier

estimator1 = RotationForest(n_estimators = 100)
estimator2 = RandomForestClassifier(n_estimators = 100)

n_estimators = 2  # the number of base estimators per cascade layer
estimators = [RotationForest(random_state = 1), RotationForest(random_state=2), RandomForestClassifier(random_state=3),RandomForestClassifier(random_state = 4)]

model.set_estimator(estimators)


In [ ]:
model.fit(trainData, trainy)

In [ ]:
# gasdata = pd.read_csv("/Users/wzhang/Downloads/Gas Sensors Data - Sheet1 (3) (1).csv")
# result = gasdata['Gas']
# gasdata = gasdata.drop(columns=['Gas'])
# gasdata= (gasdata - np.min(gasdata)) / (np.max(gasdata) - np.min(gasdata))
# images =  sorted(os.listdir("/Users/wzhang/Downloads/No Gas 4/No Gas"))
# X_dat = []
# for image in images:
#        #print(image)
#        img = mpimg.imread('/Users/wzhang/Downloads/No Gas 4/No Gas/'+image)
#        img.resize(224,224,3)
#        img = img/170
#        X_dat.append(img)
# test_images = np.array(X_dat)
# print(test_images[1])
# print(test_images.shape)

# savemodel
# testAttrX
# testImagesX

alexnetPredictTest = np.array(alexnet.predict(testImagesX))
resnetPredictTest = np.array(resnet.predict(testImagesX))
mobilenetPredictTest = np.array(mobilenet.predict(testImagesX))
print(alexnetPredictTest.shape)
print(resnetPredictTest.shape)
print(mobilenetPredictTest.shape)
print(gasdata.shape)

testData = concatenate([alexnetPredictTest, resnetPredictTest, mobilenetPredictTest, testAttrX])
print(testData)
print(testData.shape)

performance = model.predict(testData)
print(performance)
actual = []
for value in performance: 
    #print(value)
    if(value>=0.5):
            actual.append(1)
    else:
            actual.append(0)
acutal = np.array(actual)

accuracy = accuracy_score(testy, actual)
print('Accuracy: %f' % accuracy)
precision = precision_score(testy, actual)
print('Precision: %f' % precision)
recall = recall_score(testy, actual)
print('Recall: %f' % recall)
f1 = f1_score(testy, actual)
print('F1 score: %f' % f1)